# Norman example CAPRA API demo

This notebook is a commented walkthrough of the source-file CAPRA API on the bundled Norman example data. It is meant to be readable from a fresh GitHub checkout: no pip package installation is required, and all input paths are repository-relative.

What this demo does:

1. Imports CAPRA directly from the local `capra/` source directory.
2. Loads the bundled Norman AnnData subset and matching GenePT embedding subset.
3. Builds a condition-level split through `CAPRAData.register_evaluation_partitions(split_strategy="auto")`.
4. Estimates DEG references only from train/validation conditions.
5. Fits CAPRA with training defaults aligned to the benchmark wrapper, aside from using the bundled demo input files.
6. Generates counterfactual expression profiles for all held-out test conditions.

The default settings are intentionally the same scale as the benchmark wrapper (`80` epochs and `500` predicted cells per test condition). For a quick smoke test, lower `max_epochs` and `n_pred` in the configuration cell.

In [1]:
from pathlib import Path
import sys

# Make the notebook robust to being launched either from the repository root
# or from the demo/ directory. This keeps every path below repository-relative.
cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == 'demo' else cwd

# CAPRA is intentionally used as a source checkout here, not as an installed
# package. Adding capra/ to sys.path mirrors the release demo script.
source_root = repo_root / 'capra'
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

# Public API objects used in this notebook:
# - CAPRAData: data loading, canonicalization, split registration, and assets
# - CAPRA: model fitting and counterfactual prediction
# - load_gene_embedding_table: GenePT embedding loader with a control vector
from frame import CAPRA, CAPRAData, load_gene_embedding_table


In [2]:
# Reproducibility settings.
# seed follows the benchmark wrapper and controls model initialization plus
# CAPRA sampling streams. split_seed controls only the generated condition
# partition, so changing it changes train/val/test membership.
seed = 24
split_seed = 1

# Training and prediction defaults are wrapper-aligned. For local smoke tests,
# set max_epochs=1 and n_pred=16 before running the notebook.
max_epochs = 80
n_pred = 500

# Outputs are local run artifacts and should not be committed.
output_dir = repo_root / 'demo' / 'outputs' / 'norman_example_api_notebook'

# Bundled demo inputs. The AnnData file is a small Norman subset; the GenePT
# pickle contains only the embeddings needed by that subset.
adata_path = repo_root / 'demo' / 'data' / 'norman_example.h5ad'
embedding_path = repo_root / 'demo' / 'data' / 'norman_example_genept_embeddings.pkl'

print('adata_path:', adata_path.relative_to(repo_root))
print('embedding_path:', embedding_path.relative_to(repo_root))
print('adata_path exists:', adata_path.exists())
print('embedding_path exists:', embedding_path.exists())


adata_path: demo/data/norman_example.h5ad
embedding_path: demo/data/norman_example_genept_embeddings.pkl
adata_path exists: True
embedding_path exists: True


In [3]:
# CAPRAData is the stateful data-preparation object used by the public API.
# The condition_key and perturbation_key point to AnnData obs columns that
# contain perturbation labels before CAPRA canonicalization.
capra_data = CAPRAData(
    study_name='norman_example_demo',
    condition_key='condition',
    perturbation_key='perturbation',
)

# Load the bundled h5ad and create canonical CAPRA labels. Canonicalization
# removes control tokens and sorts genes inside combination labels, so A+B
# and B+A map to the same condition.
capra_data.load_single_cell_adata(adata_path)
capra_data.harmonize_perturbation_metadata(copy=False)

# Build a split from the observed condition list through the single public
# split API. split_strategy="auto" detects whether the dataset is single-gene
# only or contains double-gene perturbations. For combination datasets it
# records held-out subgroups as combo_seen0, combo_seen1, combo_seen2, and
# unseen_single. This demo does not read any precomputed split pickle.
capra_data.register_evaluation_partitions(split_strategy="auto", random_state=split_seed)

# Attach GenePT embeddings. add_control=True inserts a zero control vector
# named ctrl, matching CAPRA condition-prior construction.
capra_data.embedding_table = load_gene_embedding_table(embedding_path, add_control=True, control_label='ctrl')

# DEG references are estimated only from train+validation conditions. Test
# conditions are therefore not used to build supervised DEG masks.
capra_data.estimate_trainval_deg_reference(method='t-test')

# Build model-ready assets: control-relative expression statistics, GenePT
# neighbor priors, top-DEG masks, and dataloader metadata. The defaults here
# match the benchmark wrapper: topk_deg=100, knn_topk=5, knn_temperature=12.0.
capra_data.build_control_relative_training_state()

print('cells x genes:', capra_data.adata.shape)
print('train/val/test conditions:', len(capra_data.splits['train']), len(capra_data.splits['val']), len(capra_data.splits['test']))
print('test subgroup counts:', {key: len(value) for key, value in capra_data.subgroup['test_subgroup'].items()})


cells x genes: (8568, 5025)
train/val/test conditions: 131 15 80
test subgroup counts: {'combo_seen0': 4, 'combo_seen1': 48, 'combo_seen2': 7, 'unseen_single': 21}


In [4]:
# Fit the CAPRA response operator. Only the fields that differ per demo run
# are passed here: output location, run name, epoch count, and seed. Other
# training settings intentionally come from the CAPRA API defaults so they
# stay aligned with the benchmark wrapper, including accelerator="auto",
# mixed precision, batch size, learning rate, workers, and pin_memory.
model = CAPRA(capra_data)
model.fit_capra_response_operator(
    output_dir=output_dir / 'results',
    run_name=f'capra_response_operator_seed{split_seed}',
    n_epochs=max_epochs,
    min_epochs=min(20, int(max_epochs)),
    seed=seed,
)


CAPRA training | epoch 1/80 | train_loss=0.4024 | val_unseen_loss=0.2495 | monitor=0.3523 | early_stop=warmup(1/20)
CAPRA training | epoch 2/80 | train_loss=0.3873 | val_unseen_loss=0.2376 | monitor=0.3313 | early_stop=warmup(2/20)
CAPRA training | epoch 3/80 | train_loss=0.3794 | val_unseen_loss=0.2320 | monitor=0.3232 | early_stop=warmup(3/20)
CAPRA training | epoch 4/80 | train_loss=0.3756 | val_unseen_loss=0.2263 | monitor=0.3134 | early_stop=warmup(4/20)
CAPRA training | epoch 5/80 | train_loss=0.3739 | val_unseen_loss=0.2255 | monitor=0.3137 | early_stop=warmup(5/20)
CAPRA training | epoch 6/80 | train_loss=0.3723 | val_unseen_loss=0.2234 | monitor=0.3095 | early_stop=warmup(6/20)
CAPRA training | epoch 7/80 | train_loss=0.3712 | val_unseen_loss=0.2160 | monitor=0.2984 | early_stop=warmup(7/20)
CAPRA training | epoch 8/80 | train_loss=0.3704 | val_unseen_loss=0.2158 | monitor=0.2970 | early_stop=warmup(8/20)
CAPRA training | epoch 9/80 | train_loss=0.3694 | val_unseen_loss=0.2174

In [5]:
# Predict every held-out condition in the automatically generated test split.
# Because sampling_state is omitted, CAPRA reuses the fitted run seed, which
# is the same prediction convention used by the benchmark wrapper.
test_conditions = list(capra_data.splits['test'])
predictions = model.generate_counterfactual_profiles(
    pert_list=test_conditions,
    n_pred=int(n_pred),
)

# The prediction dictionary maps each condition to an n_pred x n_genes matrix.
# Print only a compact preview so the notebook stays readable.
print('Predicted held-out conditions:', len(test_conditions))
for condition in test_conditions[:3]:
    print(condition, predictions[condition].shape)


Predicted held-out conditions: 80
AHR+FEV (500, 5025)
ARRDC3 (500, 5025)
BAK1 (500, 5025)
